# Terrapath: recorrido del motor paso a paso

Este notebook corre el motor real del proyecto, en orden, desde el clima hasta los datos de la app.
No copia el código: lo importa de `src/`. Así el notebook y la app siempre usan la misma versión.

**Reglas que respeta todo el motor**
- No se inventan datos. Si un dato no está en la fuente, queda vacío (`None` / `NaN`) y se reporta como desconocido.
- Un dato faltante **nunca** cuenta como cero ni como favorable.
- Cada salida guarda su procedencia: fuente, archivo, SHA-256, variable, unidad, período y ubicación.

**Fuentes**: NASA POWER (clima diario), FAO-56 (Kc y ET0 Penman-Monteith), EcoCrop (límites de cada cultivo),
NASA SRTM (altitud y pendiente), NASA GPM IMERG (lluvia satelital, para comparar) y FAOSTAT (cultivo típico).

## 1. Preparación
Carga el proyecto y lista los 6 lugares piloto.

In [1]:
import json, sys, os
from pathlib import Path
import pandas as pd
import plotly.graph_objects as go

ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
os.chdir(ROOT); sys.path.insert(0, str(ROOT))
pd.set_option('display.max_columns', 30); pd.set_option('display.width', 160)

from src.climate_profile import pilots, monthly_profile, daily_series
from src.crop_library import build_library, missing_properties
from src.rotation_scores import score_point
from src.terrain import terrain_table

PILOTS = pilots(ROOT)
pd.DataFrame(PILOTS)[['name', 'label', 'latitude', 'longitude']]

,name,label,latitude,longitude
0,piura_1,"Huancabamba, Piura (PE)",-5.24,-79.45
1,piura_2,Piura sur (PE),-5.34,-79.42
2,piura_3,Piura norte (PE),-4.64,-79.72
3,iowa,Iowa central (US),42.03,-93.62
4,kenia,Nakuru (KE),-0.33,35.94
5,california,Valle Central (US),36.95,-120.28


## 2. El clima de cada lugar (NASA POWER, 1981-2024)

`monthly_profile` resume la serie diaria por mes: temperatura, días con helada (mínima bajo 0 °C), lluvia y su
variabilidad, y ET0 (el agua que "pide" la atmósfera, FAO-56 Penman-Monteith). Aquí: Iowa central.

In [2]:
iowa = next(p for p in PILOTS if p['name'] == 'iowa')
profile, meta = monthly_profile(iowa['latitude'], iowa['longitude'], ROOT)
print('Archivo:', meta['file']); print('SHA-256:', meta['sha256']); print('Período:', meta['period'])
profile[['month', 't_min_mean_c', 't_max_mean_c', 'frost_days_mean', 'rain_mean_mm', 'rain_cv', 'et0_mean_mm']].round(1)

Archivo: data/raw/power/POWER_Point_Daily_19810101_20241231_042d03N_0093d62W_LST.csv
SHA-256: 3cd5900cbf4e479a61b8840e5da601d353cd7c4e0ba71576075fcced785c8388
Período: ['1981-01-01', '2024-12-31']


,month,t_min_mean_c,t_max_mean_c,frost_days_mean,rain_mean_mm,rain_cv,et0_mean_mm
0,1,-11.5,-2.1,30.6,23.5,0.5,13.0
1,2,-9.0,1.5,27.0,25.2,0.5,22.4
2,3,-2.8,8.9,23.4,51.3,0.6,49.7
3,4,3.0,16.0,8.5,83.8,0.4,82.2
4,5,9.8,22.0,0.3,113.4,0.4,115.0
5,6,15.7,27.1,0.0,126.3,0.5,139.0
6,7,18.2,29.3,0.0,108.1,0.5,152.7
7,8,17.0,28.6,0.0,105.6,0.5,134.3
8,9,12.1,24.8,0.1,80.0,0.5,101.5
9,10,4.8,16.9,5.4,67.3,0.6,62.6


In [3]:
MESES = ['ene', 'feb', 'mar', 'abr', 'may', 'jun', 'jul', 'ago', 'sep', 'oct', 'nov', 'dic']
def barras(valores, titulo, color, unidad):
    fig = go.Figure(go.Bar(x=MESES, y=valores, marker_color=color, text=[f'{v:.0f}' for v in valores], textposition='outside'))
    fig.update_layout(title=titulo, yaxis_title=unidad, template='simple_white', height=320, margin=dict(t=50, b=30))
    return fig

barras(profile.frost_days_mean, 'Iowa central: días con helada por mes (promedio)', '#2F6F95', 'días').show()
barras(profile.rain_mean_mm, 'Iowa central: lluvia por mes (promedio)', '#1E8F6A', 'mm').show()

## 3. La biblioteca de cultivos

`build_library` junta, para cada cultivo, sus límites de EcoCrop (temperatura, lluvia, altitud, ciclo), su Kc FAO-56
(Tabla 12, verificado a mano en el PDF) y su familia botánica. **Lo que la fuente no trae queda vacío.**
Por ejemplo, maíz y soya no tienen temperatura letal de helada en EcoCrop.

In [4]:
library = build_library(ROOT)
library[['crop_key', 'family', 'family_leguminosae', 'kc_mid', 'killing_temperature_c', 't_abs_min_c', 't_abs_max_c',
         'max_altitude_m', 'cycle_min_days']]

,crop_key,family,family_leguminosae,kc_mid,killing_temperature_c,t_abs_min_c,t_abs_max_c,max_altitude_m,cycle_min_days
0,maize,Gramineae,False,1.20,NaN,10.0,47.0,4000.0,65.0
1,wheat,Gramineae,False,1.15,0.0,5.0,27.0,4500.0,90.0
2,soybean,Leguminosae,True,1.15,NaN,10.0,38.0,3000.0,75.0
3,bean_faba,Leguminosae,True,NaN,NaN,5.0,32.0,2700.0,100.0
4,potato,Solanaceae,False,1.15,0.0,7.0,30.0,2800.0,90.0
5,sorghum,Gramineae,False,NaN,NaN,8.0,40.0,2500.0,90.0
6,barley,Gramineae,False,1.15,0.0,2.0,40.0,4400.0,90.0
7,sunflower,Compositae,False,NaN,0.0,5.0,45.0,2600.0,70.0
8,lentil,Leguminosae,True,1.10,NaN,5.0,32.0,3800.0,70.0
9,tarwi,Leguminosae,True,NaN,-5.0,6.0,30.0,4000.0,150.0


In [5]:
# Qué le falta a cada cultivo (estos huecos se reportan, no se rellenan)
faltan = missing_properties(library)
pd.Series({k: ', '.join(v) if isinstance(v, (list, tuple)) else v for k, v in faltan.items()}, name='propiedades sin dato en la fuente').to_frame()

,propiedades sin dato en la fuente
maize,killing_temperature_c
wheat,
soybean,killing_temperature_c
bean_faba,killing_temperature_c
potato,
sorghum,killing_temperature_c
barley,
sunflower,
lentil,killing_temperature_c
tarwi,


## 4. Altitud y terreno (NASA SRTM, 30 m)

La celda de clima de POWER mide unos 50 km y promedia la altitud. SRTM da la altitud real del punto.
En montaña la diferencia es grande (Piura norte: la chacra a 2864 m, la celda a 878 m).

In [6]:
terreno = terrain_table(ROOT)
terreno[['pilot', 'elevation_point_m', 'elevation_mean_1km_m', 'slope_mean_1km_deg', 'slope_p90_1km_deg', 'void_cells_1km', 'file']]

,pilot,elevation_point_m,elevation_mean_1km_m,slope_mean_1km_deg,slope_p90_1km_deg,void_cells_1km,file
0,piura_1,1921.0,1953.6,13.86,25.34,0,data/raw/srtm/S06W080.SRTMGL1.hgt.zip
1,piura_2,1761.0,1770.4,11.73,20.57,0,data/raw/srtm/S06W080.SRTMGL1.hgt.zip
2,piura_3,2864.0,2776.0,15.46,23.77,0,data/raw/srtm/S05W080.SRTMGL1.hgt.zip
3,iowa,285.0,286.1,2.99,5.26,0,data/raw/srtm/N42W094.SRTMGL1.hgt.zip
4,kenia,2166.0,2166.4,2.96,5.39,0,data/raw/srtm/S01E035.SRTMGL1.hgt.zip
5,california,55.0,54.9,1.81,3.01,0,data/raw/srtm/N36W121.SRTMGL1.hgt.zip


## 5. Qué se puede sembrar y las rutas (motor en vivo, Iowa)

`score_point` hace todo el trabajo para un punto:
1. **Elegibilidad** de cada cultivo: *apto*, *excluido* (con motivo) o *desconocido* (le falta un dato).
2. Arma todas las rotaciones de 3 campañas que **no repiten familia de planta** (contando la vuelta al inicio).
3. Puntúa cada una de 0 a 100 en **Agua** (Kc × ET0 frente a la lluvia), **Tierra** (leguminosa, cambios de familia,
   cobertura) y **Cosecha segura** (helada a la temperatura letal, variabilidad de lluvia y años secos).
4. Se queda con el frente de Pareto y elige 3 rutas con cultivos distintos.

In [7]:
result = score_point(iowa['latitude'], iowa['longitude'], iowa['label'], ROOT)
print('Estado:', result['status'], '·', result['window'], '· rotaciones evaluadas:', result['rotations'])
eleg = pd.DataFrame(result['eligibility'])[['crop_key', 'state', 'failed', 'missing', 'months']]
eleg.sort_values('state')

Estado: scored · inicio may · rotaciones evaluadas: 102


,crop_key,state,failed,missing,months
1,wheat,eligible,[],[],"[5, 6, 7]"
4,potato,eligible,[],[],"[5, 6, 7]"
6,barley,eligible,[],[],"[5, 6, 7]"
7,sunflower,eligible,[],[],"[5, 6, 7]"
9,tarwi,eligible,[],[],"[5, 6, 7, 8, 9]"
12,clover_red,eligible,[],[],"[5, 6, 7, 8]"
14,alfalfa,eligible,[],[],"[5, 6, 7, 8]"
10,cassava,excluded,[frost],[],"[5, 6, 7, 8, 9, 10]"
13,rice,excluded,[annual_rain],[killing_temperature_c],"[5, 6, 7]"
0,maize,unknown,[],[killing_temperature_c],"[5, 6, 7]"


In [8]:
filas = []
for grupo in ('routes', 'routes_with_fallow'):
    for nombre, r in (result.get(grupo) or {}).items():
        if r: filas.append(dict(grupo=grupo, ruta=nombre, secuencia=' → '.join(r['sequence']),
                                agua=r['water'], tierra=r['soil'], cosecha_segura=r['risk'], faltan=r.get('missing')))
pd.DataFrame(filas)

,grupo,ruta,secuencia,agua,tierra,cosecha_segura,faltan
0,routes,ruta_agua,alfalfa → barley → potato,78.9,100.0,76.1,[]
1,routes,ruta_suelo,alfalfa → potato → wheat,78.9,100.0,76.1,[]
2,routes_with_fallow,ruta_agua,alfalfa → barley → barbecho,81.2,80.0,76.4,[]
3,routes_with_fallow,ruta_suelo,alfalfa → potato → barbecho,81.2,80.0,76.4,[]
4,routes_with_fallow,ruta_riesgo,alfalfa → wheat → barbecho,81.2,80.0,76.4,[]


In [9]:
# Rotaciones fuera de las rutas porque su agua es desconocida (cultivos sin Kc FAO-56): no cuentan como cero ni como buenas
result['unknown_water']

{'rotations': 90,
 'crop_sets': [['alfalfa', 'barley', 'sunflower'],
  ['alfalfa', 'potato', 'sunflower'],
  ['alfalfa', 'sunflower', 'wheat'],
  ['barley', 'clover_red', 'potato'],
  ['barley', 'clover_red', 'sunflower'],
  ['barley', 'potato', 'sunflower'],
  ['barley', 'potato', 'tarwi'],
  ['barley', 'sunflower', 'tarwi'],
  ['clover_red', 'potato', 'sunflower'],
  ['clover_red', 'potato', 'wheat'],
  ['clover_red', 'sunflower', 'wheat'],
  ['potato', 'sunflower', 'tarwi'],
  ['potato', 'sunflower', 'wheat'],
  ['potato', 'tarwi', 'wheat'],
  ['sunflower', 'tarwi', 'wheat']],
 'crops_without_kc': ['clover_red', 'sunflower', 'tarwi'],
 'note': 'fuera del frente y de las rutas: su agua es desconocida, no cero ni favorable'}

## 6. Las rutas de los 6 pilotos
Leídas de `data/processed/routes_<piloto>.json` (salidas del motor).

In [10]:
filas = []
for p in PILOTS:
    r = json.loads((ROOT / f"data/processed/routes_{p['name']}.json").read_text())
    for grupo, tipo in (('routes', 'tres cultivos'), ('routes_with_fallow', 'con descanso')):
        for ruta in (r.get(grupo) or {}).values():
            if ruta: filas.append(dict(lugar=p['label'], tipo=tipo, secuencia=' → '.join(ruta['sequence']),
                                       agua=ruta['water'], tierra=ruta['soil'], cosecha_segura=ruta['risk']))
    if r['status'] != 'scored': filas.append(dict(lugar=p['label'], tipo=r['status'], secuencia=r.get('reason', '')))
rutas = pd.DataFrame(filas).drop_duplicates(['lugar', 'secuencia'])
rutas

,lugar,tipo,secuencia,agua,tierra,cosecha_segura
0,"Huancabamba, Piura (PE)",tres cultivos,alfalfa → barley → potato,98.1,100.0,53.9
1,"Huancabamba, Piura (PE)",tres cultivos,alfalfa → potato → wheat,98.1,100.0,53.9
2,"Huancabamba, Piura (PE)",con descanso,alfalfa → barley → barbecho,98.6,80.0,53.5
3,"Huancabamba, Piura (PE)",con descanso,alfalfa → potato → barbecho,98.6,80.0,53.5
4,"Huancabamba, Piura (PE)",con descanso,barley → potato → barbecho,97.2,20.0,54.6
5,Piura sur (PE),tres cultivos,alfalfa → barley → potato,64.0,100.0,52.6
6,Piura sur (PE),tres cultivos,alfalfa → potato → wheat,64.0,100.0,52.6
7,Piura sur (PE),con descanso,alfalfa → barley → barbecho,65.5,80.0,52.3
8,Piura sur (PE),con descanso,alfalfa → potato → barbecho,65.5,80.0,52.3
9,Piura sur (PE),con descanso,barley → potato → barbecho,61.1,20.0,53.1


## 7. Prueba hacia atrás (backtest)

Se calibra con 1981-2015 y se prueba con 2016-2024: ¿la ruta habría pedido menos agua (déficit) que el
**maíz solo**, el cultivo típico según la superficie cosechada de FAOSTAT? Si un año no tiene dato, el total queda vacío.

In [11]:
filas = []
for p in PILOTS:
    path = ROOT / f"data/processed/backtest_{p['name']}.json"
    if not path.exists(): continue
    b = json.loads(path.read_text())
    if b.get('status') != 'tested':
        filas.append(dict(lugar=p['label'], estado=b.get('status'))); continue
    tipico = b['typical_monoculture']['crop']
    for nombre, ruta in b['routes'].items():
        if not ruta: continue
        c = b['comparisons'][nombre][tipico]['deficit_mm']
        filas.append(dict(lugar=p['label'], estado='probado', ruta=' → '.join(ruta['sequence']),
                          deficit_ruta_mm=c['route']['total'], deficit_maiz_mm=c['monoculture']['total'], veredicto=c['verdict']))
pd.DataFrame(filas)

,lugar,estado,ruta,deficit_ruta_mm,deficit_maiz_mm,veredicto
0,"Huancabamba, Piura (PE)",probado,alfalfa → barley → potato,942.1,1137.7,menos
1,"Huancabamba, Piura (PE)",probado,alfalfa → potato → wheat,942.1,1137.7,menos
2,Piura sur (PE),probado,alfalfa → barley → potato,1691.3,1808.1,menos
3,Piura sur (PE),probado,alfalfa → potato → wheat,1691.3,1808.1,menos
4,Piura norte (PE),calibration_no_valid_rotation,NaN,NaN,NaN,NaN
5,Iowa central (US),probado,alfalfa → barley → potato,1494.3,1869.8,menos
6,Iowa central (US),probado,alfalfa → potato → wheat,1494.3,1869.8,menos
7,Nakuru (KE),probado,alfalfa → barley → potato,1207.5,1772.8,menos
8,Nakuru (KE),probado,alfalfa → potato → wheat,1207.5,1772.8,menos
9,Valle Central (US),calibration_no_known_water,NaN,NaN,NaN,NaN


## 8. Sensibilidad: ¿cambian las rutas si el agua pesa más?

Se sube el peso del agua de 40 % a 50 % (y se barre de 20 % a 80 %). Resultado: las rutas no cambian en ningún piloto.

In [12]:
from src.sensitivity import sensitivity_point
s = sensitivity_point(iowa['latitude'], iowa['longitude'], iowa['label'], ROOT)
print('Pesos:', s['weights']['base'], '→', s['weights']['alternative'])
print('Cambios al barrer el peso del agua de 20 % a 80 %:', s['sweep_changes'] or 'ninguno')
pd.DataFrame({'agua 40 %': {k: ' → '.join(v) if v else '—' for k, v in s['per_axis']['base'].items()},
              'agua 50 %': {k: ' → '.join(v) if v else '—' for k, v in s['per_axis']['alternative'].items()}})

Pesos: {'water': 0.4, 'soil': 0.3, 'risk': 0.3} → {'water': 0.5, 'soil': 0.25, 'risk': 0.25}
Cambios al barrer el peso del agua de 20 % a 80 %: ninguno


,agua 40 %,agua 50 %
ruta_agua,alfalfa → barley → potato,alfalfa → barley → potato
ruta_suelo,alfalfa → potato → wheat,alfalfa → potato → wheat
ruta_riesgo,—,—


## 9. Lluvia: NASA POWER frente a GPM IMERG

El motor usa la lluvia de POWER. IMERG es otra medición satelital, más fina (0,1°). Las comparamos en 2001-2024.
En Piura, IMERG ve **aproximadamente el doble** de lluvia. Sin estaciones en tierra (SENAMHI) no se puede decir cuál acierta,
así que el motor **no se cambió**: queda como limitación declarada.

In [13]:
filas = []
for p in PILOTS:
    frame, _ = daily_series(p['latitude'], p['longitude'], ROOT)
    power = frame.set_index('fecha')['PRECTOTCORR'].resample('MS').sum()
    imerg = pd.read_csv(ROOT / f"data/processed/imerg_{p['name']}.csv", parse_dates=['period_start']).set_index('period_start').rain_mm
    both = pd.concat([imerg.rename('imerg'), power.rename('power')], axis=1, join='inner').dropna()
    both = both[both.index.year <= 2024]
    anual = both.groupby(both.index.year).sum().mean()
    filas.append(dict(lugar=p['label'], meses=len(both), imerg_mm_año=round(anual.imerg), power_mm_año=round(anual.power),
                      correlacion_mensual=round(both.corr().iloc[0, 1], 2)))
lluvia = pd.DataFrame(filas)
lluvia

,lugar,meses,imerg_mm_año,power_mm_año,correlacion_mensual
0,"Huancabamba, Piura (PE)",288,834,467,0.69
1,Piura sur (PE),288,782,310,0.72
2,Piura norte (PE),288,879,382,0.77
3,Iowa central (US),288,1025,840,0.97
4,Nakuru (KE),288,1375,1305,0.87
5,Valle Central (US),288,419,412,0.95


In [14]:
fig = go.Figure()
fig.add_bar(name='NASA POWER (lo que usa el motor)', x=lluvia.lugar, y=lluvia.power_mm_año, marker_color='#1E8F6A',
            text=lluvia.power_mm_año, textposition='outside')
fig.add_bar(name='GPM IMERG', x=lluvia.lugar, y=lluvia.imerg_mm_año, marker_color='#86B3CF',
            text=lluvia.imerg_mm_año, textposition='outside')
fig.update_layout(title='Lluvia promedio al año, 2001-2024 (mm)', barmode='group', template='simple_white', height=420,
                  legend=dict(orientation='h', y=-0.2), margin=dict(t=50))
fig.show()

## 10. Lo que recibe la app

`scripts/build_app.py` exporta todo esto a `app/data/app_data.js`. La app no calcula rutas nuevas ni rellena datos:
solo muestra lo que sale del motor. Aquí, el resumen de Iowa tal como lo ve la app.

In [15]:
texto = (ROOT / 'app/data/app_data.js').read_text()
app = json.loads(texto[texto.index('=') + 1:].rstrip().rstrip(';'))
p = next(x for x in app['pilots'] if x['id'] == 'iowa')
print('Generado:', app['generated'], '·', app['note'])
print('Altitud chacra (SRTM):', p['terrain']['srtm_m'], 'm · celda de clima (POWER):', p['terrain']['merra2_m'], 'm')
print('Primera helada:', p['first_frost'])
pd.DataFrame(p['crops'])[['key', 'state', 'months', 'rain_mm', 'demand_mm', 'water', 'risk', 'frost_evaluable']]

Generado: 2026-11-15 · Datos de tamizaje del motor Terrapath (NASA POWER, SRTM, EcoCrop, FAO-56). No es una recomendación agronómica.
Altitud chacra (SRTM): 285.0 m · celda de clima (POWER): 310.0 m
Primera helada: {'month': 10, 'day': 13, 'years_with_frost': 43, 'seasons': 43}


,key,state,months,rain_mm,demand_mm,water,risk,frost_evaluable
0,maize,unknown,"[5, 6, 7]",347.7,488.1,71.2,76.2,False
1,wheat,eligible,"[5, 6, 7]",347.7,467.8,74.3,75.6,True
2,soybean,unknown,"[5, 6, 7]",347.7,467.8,74.3,76.2,False
3,bean_faba,unknown,"[5, 6, 7, 8]",453.3,NaN,NaN,77.8,False
4,potato,eligible,"[5, 6, 7]",347.7,467.8,74.3,75.6,True
5,sorghum,unknown,"[5, 6, 7]",347.7,NaN,NaN,76.2,False
6,barley,eligible,"[5, 6, 7]",347.7,467.8,74.3,75.6,True
7,sunflower,eligible,"[5, 6, 7]",347.7,NaN,NaN,75.6,True
8,lentil,unknown,"[5, 6, 7]",347.7,447.4,77.7,76.2,False
9,tarwi,eligible,"[5, 6, 7, 8, 9]",533.3,NaN,NaN,80.4,True


## 11. Limitaciones (no se esconden)

- **Clima en celdas de ~50 km** (POWER/MERRA-2): en montaña suaviza el relieve; la app muestra un aviso de zona de montaña.
- **Maíz y soya** no tienen temperatura letal de helada en EcoCrop: su helada es desconocida y el motor no los pone en rutas.
- **Cultivos sin Kc FAO-56** (por ejemplo tarwi, trébol): su agua es desconocida; quedan fuera de las rutas.
- **Lluvia**: POWER e IMERG no coinciden en Piura; falta validar con estaciones.
- **Sin rendimientos por chacra**: por eso el eje se llama "Cosecha segura" y no "más cosecha".
- **Calendario**: meses de campaña según el ciclo mínimo de EcoCrop, no el calendario local.
- Es una herramienta de **tamizaje**: no reemplaza a un técnico agrario.